# Task 1: Analyse Example Data
**Group 1 Metric**: Bennett, Alpert, and Goldstein’s S


In [1]:
import json
import glob
import os
import pandas as pd

# load example data files
files = sorted(glob.glob("example_data/*.jsonl"))
if not files:
    # fallback to server path if needed
    files = sorted(glob.glob("/scratch/group_01/bsem-group/annotation_experiment/example_data/*.jsonl"))

data = []
for f in files:
    with open(f, "r", encoding="utf-8") as fp:
        for line in fp:
            line = line.strip()
            if line:
                data.append(json.loads(line))

df = pd.DataFrame(data)
print(f"Total samples loaded: {len(df)}")


Total samples loaded: 81


## 1. Check for Missing Data


In [2]:
# check if any samples have missing annotators or missing labels
missing_samples = 0

for _, row in df.iterrows():
    anns = row.get("labels_annotation", [])
    # check that both anna and max annotated
    if len(anns) != 2:
        missing_samples += 1

print(f"Samples with missing annotations: {missing_samples} (out of {len(df)})")
print(f"Batches in dataset: {list(df['batch_number'].unique())}")


Samples with missing annotations: 0 (out of 81)
Batches in dataset: [14, 43]


## 2. 2x2 Matrix: "offensive" vs "not offensive"


In [3]:
# extract labels for anna and max
def get_user_label(labels_list, user):
    for entry in labels_list:
        if entry.get("user") == user:
            lbls = entry.get("labels", [])
            return "offensive" if "offensive" in lbls else "not offensive"
    return "missing"

df["anna"] = df["labels_annotation"].apply(lambda lst: get_user_label(lst, "anna"))
df["max"] = df["labels_annotation"].apply(lambda lst: get_user_label(lst, "max"))

# 2x2 contingency matrix (All Data)
print("=== 2x2 Matrix (All Data, N = 81) ===")
matrix_all = pd.crosstab(df["anna"], df["max"], rownames=["Anna"], colnames=["Max"], margins=True)
print(matrix_all)

# batch breakdown
print("\n=== 2x2 Matrix (Batch 14, N = 30) ===")
print(pd.crosstab(df[df["batch_number"] == 14]["anna"], df[df["batch_number"] == 14]["max"], rownames=["Anna"], colnames=["Max"], margins=True))

print("\n=== 2x2 Matrix (Batch 43, N = 51) ===")
print(pd.crosstab(df[df["batch_number"] == 43]["anna"], df[df["batch_number"] == 43]["max"], rownames=["Anna"], colnames=["Max"], margins=True))


=== 2x2 Matrix (All Data, N = 81) ===
Max            not offensive  offensive  All
Anna                                        
not offensive             62          5   67
offensive                  2         12   14
All                       64         17   81

=== 2x2 Matrix (Batch 14, N = 30) ===
Max            not offensive  offensive  All
Anna                                        
not offensive             21          4   25
offensive                  1          4    5
All                       22          8   30

=== 2x2 Matrix (Batch 43, N = 51) ===
Max            not offensive  offensive  All
Anna                                        
not offensive             41          1   42
offensive                  1          8    9
All                       42          9   51


## 3. Percentage Agreement & Bennett, Alpert, and Goldstein’s S

- **Percentage Agreement ($P_o$)**:
  $$P_o = \frac{\text{Number of Agreed Samples}}{N}$$

- **Bennett, Alpert, and Goldstein’s S**:
  $$S = 1 - \frac{k(1 - P_o)}{k - 1}$$
  For $k = 2$ categories (offensive / not offensive):
  $$S = 1 - \frac{2(1 - P_o)}{2 - 1} = 2P_o - 1$$


In [4]:
def compute_metrics(subset, name="All Data"):
    n = len(subset)
    agreements = (subset["anna"] == subset["max"]).sum()
    po = agreements / n
    
    # Bennett's S with k=2
    k = 2
    s = 1.0 - (k * (1.0 - po)) / (k - 1.0)
    
    return {
        "Subset": name,
        "N": n,
        "Agreements": agreements,
        "Disagreements": n - agreements,
        "Percentage Agreement (Po)": f"{po:.4f} ({po:.2%})",
        "Bennett's S": f"{s:.4f} ({s:.2%})"
    }

results = [
    compute_metrics(df, "All Data"),
    compute_metrics(df[df["batch_number"] == 14], "Batch 14"),
    compute_metrics(df[df["batch_number"] == 43], "Batch 43")
]

summary_df = pd.DataFrame(results)
summary_df


,Subset,N,Agreements,Disagreements,Percentage Agreement (Po),Bennett's S
0,All Data,81,74,7,0.9136 (91.36%),0.8272 (82.72%)
1,Batch 14,30,25,5,0.8333 (83.33%),0.6667 (66.67%)
2,Batch 43,51,49,2,0.9608 (96.08%),0.9216 (92.16%)
